In [1]:
import pandas as pd
import numpy as np

In [2]:
data = {
    'session_id': [1, 2, 3, 4, 5, 6, 7],
    'date': ['2026-03-20', '2026-03-21', None, '2026-03-23', '2026-03-24', '2026-03-25', '2026-03-26'],
    'utilisateur': ['Admin', 'Etudiant_1', 'Etudiant_2', None, 'Etudiant_1', 'Etudiant_3', 'Etudiant_1'],
    'debit_mbps': [100, 15, 12, 18, 9999, 14, 16], # 9999 est une erreur de capteur
    'type_connexion': ['Fibre', 'WiFi', 'WiFi', 'WiFi', 'Fibre', None, 'WiFi']
}

df = pd.DataFrame(data)

print("Données à nettoyer ")
print(df)

Données à nettoyer 
   session_id        date utilisateur  debit_mbps type_connexion
0           1  2026-03-20       Admin         100          Fibre
1           2  2026-03-21  Etudiant_1          15           WiFi
2           3         NaN  Etudiant_2          12           WiFi
3           4  2026-03-23         NaN          18           WiFi
4           5  2026-03-24  Etudiant_1        9999          Fibre
5           6  2026-03-25  Etudiant_3          14            NaN
6           7  2026-03-26  Etudiant_1          16           WiFi


# Pour resoudre le pbm de date manquante je vais utilisé la methode d'interpollation , 

In [3]:
# Mettre d'abord les date en type datetime
df['date'] =  pd.to_datetime(df['date'])

#Definir la date comme index du dataSet, naicessaire pour l'interpolation time
df = df.set_index('date')

#Création des dates manquante , on utilise .asfreq() ou .reindex() pour combler le trou pas pour donner une nouvelle ligne
df = df.asfreq('D') # 'D' pour le journalier et 'H' pour le horaire 

print("Data avec date propre")
print(df)

Data avec date propre
            session_id utilisateur  debit_mbps type_connexion
date                                                         
2026-03-20         1.0       Admin       100.0          Fibre
2026-03-21         2.0  Etudiant_1        15.0           WiFi
2026-03-22         NaN         NaN         NaN            NaN
2026-03-23         4.0         NaN        18.0           WiFi
2026-03-24         5.0  Etudiant_1      9999.0          Fibre
2026-03-25         6.0  Etudiant_3        14.0            NaN
2026-03-26         7.0  Etudiant_1        16.0           WiFi


# Pour résoudre le problème de debit_mbps

In [4]:
# On isole d'addord le outlier 999 en NaN afin de mieux calculer la moyenne 
df['debit_mbps'] = df['debit_mbps'].replace(9999,pd.NA).dropna()
print(df)
print(df.info())

            session_id utilisateur debit_mbps type_connexion
date                                                        
2026-03-20         1.0       Admin      100.0          Fibre
2026-03-21         2.0  Etudiant_1       15.0           WiFi
2026-03-22         NaN         NaN        NaN            NaN
2026-03-23         4.0         NaN       18.0           WiFi
2026-03-24         5.0  Etudiant_1        NaN          Fibre
2026-03-25         6.0  Etudiant_3       14.0            NaN
2026-03-26         7.0  Etudiant_1       16.0           WiFi
<class 'pandas.DataFrame'>
DatetimeIndex: 7 entries, 2026-03-20 to 2026-03-26
Freq: D
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   session_id      6 non-null      float64
 1   utilisateur     5 non-null      str    
 2   debit_mbps      5 non-null      object 
 3   type_connexion  5 non-null      str    
dtypes: float64(1), object(1), str(2)
memory usage: 280.0+ byte

In [ ]:
#On cherche maintenant le cv : la coefficient de variation , la différence entre la moyenne et l'ecartype, 
# si cv < 0.15 , c'est normale on prend la moyenne et sinon on prend la médiane
# Formule cv = ecarttype (std) / moyenne (mean)

# Force le type numérique réel
df['debit_mbps'] = df['debit_mbps'].astype(float)

cv = df['debit_mbps'].std() / df['debit_mbps'].mean()
print(f"Coefficient de variation: {cv:.2%}")
if(cv < 0.15):
    print("Remplacer par la moyenne (Données stable)")
else:
    print("Remplacer par la mediane (données dispercées)")


print(df)
print(df.describe())

Coefficient de variation: 115.664918%
Remplacer par la mediane (données dispercées)
            session_id utilisateur  debit_mbps type_connexion
date                                                         
2026-03-20         1.0       Admin       100.0          Fibre
2026-03-21         2.0  Etudiant_1        15.0           WiFi
2026-03-22         NaN         NaN         NaN            NaN
2026-03-23         4.0         NaN        18.0           WiFi
2026-03-24         5.0  Etudiant_1         NaN          Fibre
2026-03-25         6.0  Etudiant_3        14.0            NaN
2026-03-26         7.0  Etudiant_1        16.0           WiFi
       session_id  debit_mbps
count    6.000000    5.000000
mean     4.166667   32.600000
std      2.316607   37.706763
min      1.000000   14.000000
25%      2.500000   15.000000
50%      4.500000   16.000000
75%      5.750000   18.000000
max      7.000000  100.000000


In [6]:
# 2. On remplace les NaN par cette médiane
df['debit_mbps'] = df['debit_mbps'].fillna(df['debit_mbps'].median())
# 3. On vérifie le résultat
print(f"Valeur utilisée pour le remplissage : {df['debit_mbps'].median()}")
print(df)

Valeur utilisée pour le remplissage : 16.0
            session_id utilisateur  debit_mbps type_connexion
date                                                         
2026-03-20         1.0       Admin       100.0          Fibre
2026-03-21         2.0  Etudiant_1        15.0           WiFi
2026-03-22         NaN         NaN        16.0            NaN
2026-03-23         4.0         NaN        18.0           WiFi
2026-03-24         5.0  Etudiant_1        16.0          Fibre
2026-03-25         6.0  Etudiant_3        14.0            NaN
2026-03-26         7.0  Etudiant_1        16.0           WiFi


# Pour resoudre le nom manquant on supprime la ligne car on ne peut pas inventé un nom

In [7]:
df = df.dropna(subset=['utilisateur'])
print(df)

            session_id utilisateur  debit_mbps type_connexion
date                                                         
2026-03-20         1.0       Admin       100.0          Fibre
2026-03-21         2.0  Etudiant_1        15.0           WiFi
2026-03-24         5.0  Etudiant_1        16.0          Fibre
2026-03-25         6.0  Etudiant_3        14.0            NaN
2026-03-26         7.0  Etudiant_1        16.0           WiFi


# Pour résoudre le problème de catégorie manquante on va cherché le mode et le remplaé par

In [8]:
#On precise [0] car le mode renvoie une liste
type_connexion_mode = df['type_connexion'].mode()[0]
print(f"Voici le mode : {df['type_connexion'].mode()[0]}")
#On remplace tous les type de connexion manquante
df['type_connexion'] = df['type_connexion'].fillna(type_connexion_mode)
print(df)

Voici le mode : Fibre
            session_id utilisateur  debit_mbps type_connexion
date                                                         
2026-03-20         1.0       Admin       100.0          Fibre
2026-03-21         2.0  Etudiant_1        15.0           WiFi
2026-03-24         5.0  Etudiant_1        16.0          Fibre
2026-03-25         6.0  Etudiant_3        14.0          Fibre
2026-03-26         7.0  Etudiant_1        16.0           WiFi


# Question , pour resoudre les nom on les a supprimé, mais cela affecte le mode de type de connexion. Alors comment savoir si on doit d'abord trouver la mode avant de supprimer le nom ou l'inverse ?

1. Priorité à la suppression des lignes "irrécupérables"
On commence généralement par supprimer les lignes où la donnée manquante est vitale et ne peut pas être devinée (comme l'utilisateur).

Pourquoi ? Si tu calcules le mode (la valeur la plus fréquente) avant de supprimer les lignes inutiles, tu inclus dans ton calcul des données que tu vas de toute façon jeter. C'est comme compter des votes de personnes qui n'ont pas le droit de voter.

Impact sur ton cas : En supprimant d'abord les utilisateurs inconnus, tu as "nettoyé" ta base de calcul pour le mode. Le mode que tu trouves après est donc celui qui est le plus représentatif de tes utilisateurs réels.

Résumé de la Pipeline Idéale
Casting : Convertir les types (Dates, Chiffres).

Outliers : Isoler/remplacer les valeurs aberrantes (ton 9999 -> NaN).

Filtrage : Supprimer les lignes avec des identifiants manquants (utilisateur).

Imputation : Remplir les colonnes restantes avec le mode, la médiane ou la moyenne calculés sur ce qui reste.